# MedDec Phase 3 — LLM Span Extraction

This notebook runs the zero-shot and one-shot LLM pipelines and evaluates them.
It is **independent of Phase 2** — you do not need a trained ELECTRA checkpoint.

### Can I run this locally instead of Colab?

| Model | VRAM | Free T4 | Local CPU | Notes |
|---|---|---|---|---|
| `flan-t5-small` | ~0.3 GB | ✓ fast | ✓ slow (~3 h) | Good for pipeline testing |
| `flan-t5-base` | ~1 GB | ✓ fast | possible (very slow) | Slightly better quality |
| `flan-t5-xl` | ~12 GB | ✓ (fits T4) | ✗ | Best free-tier quality |
| `Llama-3-8B` | ~16 GB | ✗ | ✗ | Needs Colab Pro / A100 |

**Recommendation**: `flan-t5-xl` on Colab free T4 gives the best results without paying.
For a quick pipeline test locally, use `flan-t5-small` and limit to 3–5 notes.

### FLAN-T5 context limit
FLAN-T5 was trained with 512 input tokens. We extend the source limit to 1024,
but notes longer than that will be truncated. Llama-3 handles 8192 tokens and won't truncate.

## Step 0 — Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
# sentencepiece is required for FLAN-T5 tokeniser
# accelerate is required for device_map='auto' (Llama-3)
!pip install -q transformers sentencepiece accelerate

In [ ]:
import shutil, sys
from pathlib import Path

# ── Edit these paths ─────────────────────────────────────────────────────────
DRIVE_CODE_DIR = Path("/content/drive/MyDrive/AI4H-project-rework/04 Code/04 Code/med-decision-extraction")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/AI4H-project-rework/02 Data")
# ─────────────────────────────────────────────────────────────────────────────

LOCAL_CODE = Path("/content/meddec")
if LOCAL_CODE.exists():
    shutil.rmtree(LOCAL_CODE)
shutil.copytree(DRIVE_CODE_DIR, LOCAL_CODE)
sys.path.insert(0, str(LOCAL_CODE))

MEDDEC_DIR  = DRIVE_DATA_DIR / "meddec-mimic-iii"
SPLITS_DIR  = MEDDEC_DIR / "splits"
GENS_DIR    = DRIVE_DATA_DIR / "gens"   # predictions saved here (persisted to Drive)
GENS_DIR.mkdir(parents=True, exist_ok=True)

print("Code :", LOCAL_CODE)
print("Data :", MEDDEC_DIR)

## Step 1 — Verify GPU + data

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

n_json = len(list((MEDDEC_DIR / "data").glob("*.json")))
n_txt  = len(list((MEDDEC_DIR / "raw_text").glob("*.txt")))
print(f"\nJSON annotations : {n_json}")
print(f"Raw text files   : {n_txt}")
print(f"Test split size  : {len((SPLITS_DIR / 'test.txt').read_text().splitlines())} notes")

## Step 2 — Choose model

Uncomment exactly one `MODEL_NAME` line.

In [ ]:
# ── Choose ONE ───────────────────────────────────────────────────────────────
MODEL_NAME = "google/flan-t5-small"   # ~0.3 GB  — fast test, weakest quality
# MODEL_NAME = "google/flan-t5-base"  # ~1 GB    — better quality, still fast on T4
# MODEL_NAME = "google/flan-t5-xl"    # ~12 GB   — best free-tier quality
# MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"  # ~16 GB — needs A100 + HF token
# ─────────────────────────────────────────────────────────────────────────────

# For Llama-3, set your HuggingFace token (request access at huggingface.co first)
# import os; os.environ["HF_TOKEN"] = "hf_..."

print(f"Selected model: {MODEL_NAME}")

## Step 3 — Zero-shot generation

For each note × 9 categories, the model is asked:
> *"Extract all substrings that represent medical decisions of category X. Print each on a new line."*

No examples are given — the model must rely entirely on its pre-training.

**Runtime estimate on T4:**
- `flan-t5-small`: ~15 min for 41 test notes (9 categories × 41 notes = 369 calls)
- `flan-t5-xl`: ~40–60 min

The pipeline saves results as it goes, so it can be safely resumed if interrupted.

In [ ]:
from gen_span_detection import run_pipeline

ZERO_SHOT_DIR = GENS_DIR / "test_zero_shot"

run_pipeline(
    meddec_dir = MEDDEC_DIR,
    splits_dir = SPLITS_DIR,
    model_name = MODEL_NAME,
    output_dir = ZERO_SHOT_DIR,
    split      = "test",
    mode       = "zero_shot",
    # max_samples = 5,   # uncomment to process only 5 notes as a quick test
)

## Step 4 — One-shot generation

Same pipeline, but each prompt now includes one worked example drawn from the **same note's annotations**.  
The example category is whichever category has the most spans in that note (always different from the target category).  

This shows the model what a decision span looks like in *this specific clinical writing style*, which often improves recall.

In [ ]:
from gen_span_detection import run_pipeline

ONE_SHOT_DIR = GENS_DIR / "test_one_shot"

run_pipeline(
    meddec_dir = MEDDEC_DIR,
    splits_dir = SPLITS_DIR,
    model_name = MODEL_NAME,
    output_dir = ONE_SHOT_DIR,
    split      = "test",
    mode       = "one_shot",
    # max_samples = 5,
)

## Step 5 — Inspect sample outputs

In [ ]:
import json
from pathlib import Path
from gen_span_detection import CATEGORY_DESCRIPTIONS

# Show predictions for one note, both modes
sample_file = sorted(ZERO_SHOT_DIR.glob("*.json"))[0]
stem        = sample_file.stem

zero_preds = json.loads(sample_file.read_text())["predictions"]
one_preds  = json.loads((ONE_SHOT_DIR / sample_file.name).read_text())["predictions"] \
             if (ONE_SHOT_DIR / sample_file.name).exists() else {}

# Also load gold
gold_data    = json.loads((MEDDEC_DIR / "data" / f"{stem}.json").read_text())
gold_by_cat  = {}
for ann in gold_data.get("annotations", []):
    from gen_span_detection import _parse_cat_number
    cat = _parse_cat_number(ann.get("category", ""))
    if cat and cat <= 9:
        gold_by_cat.setdefault(str(cat), []).append(ann["decision"])

print(f"Note: {stem}\n")
for cat in range(1, 10):
    gold   = gold_by_cat.get(str(cat), [])
    zero   = zero_preds.get(str(cat), [])
    one    = one_preds.get(str(cat), [])
    if not gold and not zero and not one:
        continue
    cat_name = CATEGORY_DESCRIPTIONS[cat-1].split(":")[0]
    print(f"--- Cat {cat}: {cat_name} ---")
    print(f"  Gold     : {gold[:3]}")
    print(f"  Zero-shot: {zero[:3]}")
    print(f"  One-shot : {one[:3]}")
    print()

## Step 6 — Evaluation

String-level F1: a prediction is a TP if it matches a gold decision string.  
Two methods are compared:
- **em** (exact match): `pred.strip() == gold.strip()`
- **approx-m**: one is a substring of the other AND word count difference ≤ 10

In [ ]:
from eval_gen import evaluate_predictions, print_results

TEST_SPLIT_FILE = SPLITS_DIR / "test.txt"

results_zero_em     = evaluate_predictions(MEDDEC_DIR, TEST_SPLIT_FILE, ZERO_SHOT_DIR, method="em")
results_zero_approx = evaluate_predictions(MEDDEC_DIR, TEST_SPLIT_FILE, ZERO_SHOT_DIR, method="approx-m")
results_one_em      = evaluate_predictions(MEDDEC_DIR, TEST_SPLIT_FILE, ONE_SHOT_DIR,  method="em")
results_one_approx  = evaluate_predictions(MEDDEC_DIR, TEST_SPLIT_FILE, ONE_SHOT_DIR,  method="approx-m")

print_results(results_zero_em,     label="Zero-shot | EM")
print_results(results_zero_approx, label="Zero-shot | Approx-M")
print_results(results_one_em,      label="One-shot  | EM")
print_results(results_one_approx,  label="One-shot  | Approx-M")

## Step 7 — README figures

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

CATEGORY_NAMES_SHORT = [
    "Contact", "Gather info", "Define problem", "Treatment goal",
    "Drug", "Therapeutic", "Eval test", "Deferment", "Advice",
]

# Zero-shot vs one-shot F1 per category (exact match)
zero_f1 = [results_zero_em["per_cat"][c]["f1"] for c in range(1, 10)]
one_f1  = [results_one_em["per_cat"][c]["f1"]  for c in range(1, 10)]

x = np.arange(9)
w = 0.36

fig_cmp, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w/2, zero_f1, w, label="Zero-shot", color="#1976D2", alpha=0.88)
ax.bar(x + w/2, one_f1,  w, label="One-shot",  color="#F57C00", alpha=0.88)

ax.axhline(results_zero_em["overall"]["f1"], color="#1976D2", linestyle="--", linewidth=1,
           label=f"Zero-shot overall F1 = {results_zero_em['overall']['f1']:.3f}")
ax.axhline(results_one_em["overall"]["f1"],  color="#F57C00", linestyle="--", linewidth=1,
           label=f"One-shot overall F1  = {results_one_em['overall']['f1']:.3f}")

ax.set_xticks(x)
ax.set_xticklabels(CATEGORY_NAMES_SHORT, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Span F1 (exact match)")
ax.set_ylim(0, 1.05)
ax.set_title(f"Zero-shot vs One-shot Span F1 per Category — {MODEL_NAME.split('/')[-1]}")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# EM vs Approx-M for one-shot (shows how strict the exact-match metric is)
one_em_f1     = [results_one_em["per_cat"][c]["f1"]     for c in range(1, 10)]
one_approx_f1 = [results_one_approx["per_cat"][c]["f1"] for c in range(1, 10)]

fig_method, ax2 = plt.subplots(figsize=(13, 5))
ax2.bar(x - w/2, one_em_f1,     w, label="Exact match",   color="#388E3C", alpha=0.88)
ax2.bar(x + w/2, one_approx_f1, w, label="Approx match",  color="#8BC34A", alpha=0.88)

ax2.set_xticks(x)
ax2.set_xticklabels(CATEGORY_NAMES_SHORT, rotation=30, ha="right", fontsize=9)
ax2.set_ylabel("Span F1")
ax2.set_ylim(0, 1.05)
ax2.set_title("Exact Match vs Approximate Match F1 (One-shot)")
ax2.legend()
ax2.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# README-ready markdown table comparing all four conditions

header = "| Category | Zero EM | Zero Approx | One EM | One Approx |"
sep    = "|---|---|---|---|---|"
print(header)
print(sep)
for c in range(1, 10):
    ze = results_zero_em["per_cat"][c]["f1"]
    za = results_zero_approx["per_cat"][c]["f1"]
    oe = results_one_em["per_cat"][c]["f1"]
    oa = results_one_approx["per_cat"][c]["f1"]
    print(f"| {CATEGORY_NAMES_SHORT[c-1]} | {ze:.3f} | {za:.3f} | {oe:.3f} | {oa:.3f} |")
print(sep)
ze = results_zero_em["overall"]["f1"]
za = results_zero_approx["overall"]["f1"]
oe = results_one_em["overall"]["f1"]
oa = results_one_approx["overall"]["f1"]
print(f"| **Overall** | **{ze:.3f}** | **{za:.3f}** | **{oe:.3f}** | **{oa:.3f}** |")

In [ ]:
DRIVE_FIGS_DIR = DRIVE_DATA_DIR / "readme_figures"
DRIVE_FIGS_DIR.mkdir(parents=True, exist_ok=True)

saves = {
    "phase3_zero_vs_one_shot.png": fig_cmp,
    "phase3_em_vs_approx.png":     fig_method,
}
for fname, fig in saves.items():
    dest = DRIVE_FIGS_DIR / fname
    fig.savefig(dest, dpi=150, bbox_inches="tight")
    print(f"Saved: {dest}")